# Meilenstein 1: Daten-Integration und Qualitätssicherung
## Projekt: Predictive Analytics im Radsport – Grand Tour Stage Ranking

### Einleitung
Dieses Notebook bildet den ersten Schritt der Datenverarbeitungspipeline. Ziel ist es, die separat erhobenen Datensätze der **Renn-Ergebnisse** (Grand Tour Stages) und der **Fahrer-Biometrie** (Physiologische Daten & Spezialisierungen) zu einem konsistenten Master-Datensatz zu verschmelzen. (Quellen anfügen?!)

Ein präzises Ranking-Modell erfordert nicht nur die historischen Platzierungen, sondern auch das physische Profil der Athleten (Größe, Gewicht, Alter) sowie deren spezifische Stärken (Climbing, Time Trial, etc.). In diesem Prozess identifizieren wir zudem Datenlücken und bereinigen den Datensatz von Redundanzen, um eine saubere Grundlage für das spätere Feature Engineering und das Training des **XGBRanker-Modells** zu schaffen.

### Arbeitsschritte in diesem Notebook:
1. **Daten-Import:** Laden der Rohdaten im JSONL-Format.
2. **Vorbereitung Data Merging**
3. **Daten-Merging und Daten in  Long Format bringen:** Verknüpfung der Tabellen via `rider_url` + Genestede Daten in Long Format bringen.
4. **Data Cleaning:** Entfernung technisch irrelevanter Merkmale (z.B. Bild-URLs).
5. **Explorative Lücken-Analyse:** Identifikation und Visualisierung fehlender Werte (`NaN`).
6. **Checkpointing:** Speicherung des bereinigten Zustands als persistente Pickle-Datei.

---

### 1. Daten Import:
Laden der Rohdaten im JSONL Format

In [1]:
# Laden der nötigen Packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [2]:
# 0. Test Directory
#print("Aktuelles Verzeichnis:", os.getcwd())

# 1. DATEN-IMPORT
# Pfade definieren (Passe diese an, falls deine Dateien in Unterordnern liegen)
results_file = '../../data/raw/grand_tour_stage_results.jsonl'
riders_file = '../../data/raw/riders_full_biometrics.jsonl'

df_results = pd.read_json(results_file, lines=True)
df_riders = pd.read_json(riders_file, lines=True)

print(df_results.head(5))
print(100*"=")
print(df_riders['url_name'].head(10))


             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   

                                            metadata  \
0  {'departure': 'Fromentine', 'arrival': 'Noirmo...   
1  {'departure': 'Challans', 'arrival': 'Les Essa...   
2  {'departure': 'La Châtaigneraie', 'arrival': '...   
3  {'departure': 'Tours', 'arrival': 'Blois', 'di...   
4  {'departure': 'Chambord', 'arrival': 'Montargi...   

                                             results  
0  [{'rank': '1', 'rider_url': 'david-zabriskie',...  
1  [{'rank': '1', 'rider_url': 'tom-boonen', 'tim...  
2  [{'rank': '1', 'rider_url'

### 2. Vorbereitung Data Merging
- Explode (Zeilen-Vervielfältigung): Die Liste der Platzierungen in der Spalte stage_results wird "gesprengt", sodass aus einer Zeile pro Etappe ca. 150 bis 180 einzelne Zeilen werden – eine für jedes Fahrergebnis.
- Spalten Extraktion: Die in den Zeilen enthaltenen Dictionaries (mit Infos wie Platzierung, Zeit und Fahrer-URL) werden in flache, eigenständige Spalten umgewandelt, damit sie mathematisch auswertbar sind.
-Merge-Vorbereitung: Die extrahierte rider_url wird als eindeutiger Schlüssel (Key) genutzt, um im nächsten Schritt die körperlichen Merkmale aus der Biometrie-Tabelle passgenau an jedes einzelne Ergebnis anzuspielen.

In [3]:
df_results_long = df_results.explode('results').reset_index(drop=True) #Reset Index damit nicht alle Stage 1 Platzierungen gleichen Index haben
#print(df_results_long.head(5))
#Nun jeder Rank für jede Etappe eine Zeile

# Extraktion der Result Daten in einzelne Spalten
results_flat = pd.json_normalize(df_results_long['results'])
#results_flat.head(5)


# Zusammenführen der df_results_long (ohne results) mit results_flat
df_results_final = pd.concat([df_results_long.drop(columns=['results']),
                              results_flat], axis = 1)

df_results_final.head(10)

,race,year,url,metadata,rank,rider_url,time_gap
0,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",1,david-zabriskie,20:5120:51
1,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",2,lance-armstrong,0:020:02
2,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",3,alexandr-vinokourov,0:530:53
3,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",4,george-hincapie,0:570:57
4,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",5,laszlo-bodrogi,0:590:59
5,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",6,floyd-landis,1:021:02
6,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",7,fabian-cancellara,",,1:02"
7,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",8,jens-voigt,1:041:04
8,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",9,vladimir-karpets,1:051:05
9,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",10,igor-gonzalez-de-galdeano,1:061:06


### 3. Daten-Merging und Daten in  Long Format bringen
- Tabellen-Merge (Left Join): Die Rennergebnisse werden über die rider_url mit der Biometrie-Tabelle verknüpft, sodass jedes Ergebnis um die Fahrer-Infos (Größe, Gewicht, etc.) ergänzt wird.


In [4]:
df_final = pd.merge(
    df_results_final,
    df_riders,
    left_on='rider_url',
    right_on='url_name',
    how='left'
)

print(df_final.head(5))

# Prüfung der fehlenden 88 Fahrerprofile
missing_rows = df_final['height'].isnull().sum()
print()

print(df_final.shape[0])
print(missing_rows)

# somit 32731 zeilen vorhanden
# 332 Zeilen nicht vorhanden -> Fahrerprofile fehlen


print(50*"=" + "\n")

# Ausgabe aller Spaltennamen

print(df_final.columns.to_list())

             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   

                                            metadata rank  \
0  {'departure': 'Fromentine', 'arrival': 'Noirmo...    1   
1  {'departure': 'Fromentine', 'arrival': 'Noirmo...    2   
2  {'departure': 'Fromentine', 'arrival': 'Noirmo...    3   
3  {'departure': 'Fromentine', 'arrival': 'Noirmo...    4   
4  {'departure': 'Fromentine', 'arrival': 'Noirmo...    5   

             rider_url    time_gap   birthdate  height  \
0      david-zabriskie  20:5120:51   1979-1-12    1.83   
1      lance-armstrong    0:020:02   1971-9-18  

- im nächsten Schritt die verbleibenden genesteden Daten in Long Format bringen

In [5]:
# metadata Spalte
df_meta = df_final['metadata'].apply(pd.Series)
#print(df_meta.head(10))

# zusammenführen mit dem Main Datensatz

df_final = pd.concat([df_final.drop(columns=['metadata']), df_meta], axis=1)
print(df_final.head(10))

             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
5  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
6  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
7  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
8  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
9  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   

  rank                  rider_url    time_gap   birthdate  height  \
0    1            david-zabriskie  20:5120:51   1979-1-12    1.83   
1  

In [6]:
#Spezialitätenprofil entpacken
df_spec = df_final['points_per_speciality'].apply(pd.Series).fillna(0)
df_final = pd.concat([df_final.drop(columns=['points_per_speciality']), df_spec], axis=1)
print(df_final.head(10))

             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
5  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
6  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
7  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
8  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
9  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   

  rank                  rider_url    time_gap   birthdate  height  \
0    1            david-zabriskie  20:5120:51   1979-1-12    1.83   
1  

In [8]:

print(df_final['won_how'].unique())
print(df_final.columns.to_list())
df_final

<ArrowStringArray>
[           'Time trial', 'Sprint of large group',           '0.9 km solo',
         'Sprint à deux',            '76 km solo',            '37 km solo',
             '? km solo', 'Sprint of small group',             '1 km solo',
           '1.5 km solo',
 ...
          '71.1 km solo',          '87.3 km solo',   'Sprint of 11 riders',
    'Sprint of 8 riders',          '27.6 km solo',   'Sprint of 33 riders',
          '58.3 km solo',          '10.1 km solo',          '15.9 km solo',
    'Sprint of 9 riders']
Length: 206, dtype: str
['race', 'year', 'url', 'rank', 'rider_url', 'time_gap', 'birthdate', 'height', 'image_url', 'name', 'nationality', 'place_of_birth', 'points_per_season_history', 'season_results', 'teams_history', 'weight', 'url_name', 'grand_tour_history', 'departure', 'arrival', 'distance', 'vertical_meters', 'profile_score', 'won_how', 'avg_speed', 'race_ranking', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 0]


,race,year,url,rank,rider_url,time_gap,birthdate,height,image_url,name,...,won_how,avg_speed,race_ranking,one_day_races,gc,time_trial,sprint,climber,hills,0
0,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,1,david-zabriskie,20:5120:51,1979-1-12,1.83,NaN,David Zabriskie,...,Time trial,54.676 km/h,n/a,31.0,1075.0,5000.0,86.0,644.0,244.0,0.0
1,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,2,lance-armstrong,0:020:02,1971-9-18,1.78,NaN,Lance Armstrong,...,Time trial,54.676 km/h,n/a,3135.0,1410.0,8149.0,258.0,6159.0,1318.0,0.0
2,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,3,alexandr-vinokourov,0:530:53,1973-9-16,1.77,NaN,Alexandr Vinokurov,...,Time trial,54.676 km/h,n/a,3038.0,5831.0,5198.0,356.0,5218.0,1322.0,0.0
3,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,4,george-hincapie,0:570:57,1973-6-29,1.91,NaN,George Hincapie,...,Time trial,54.676 km/h,n/a,3872.0,2018.0,3532.0,1722.0,1233.0,1374.0,0.0
4,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,5,laszlo-bodrogi,0:590:59,1976-12-11,1.87,images/riders/cw/ee/laszlo-bodrogi-2012.jpg,László Bodrogi,...,Time trial,54.676 km/h,n/a,700.0,1025.0,5524.0,200.0,246.0,187.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32726,vuelta-a-espana,2025,https://www.procyclingstats.com/race/vuelta-a-...,NR,mads-pedersen,",,0:00",1995-12-18,1.80,images/riders/my/dq/mads-pedersen-2026.jpg,Mads Pedersen,...,-,-,11,5070.0,1697.0,1915.0,2954.0,473.0,3770.0,0.0
32727,vuelta-a-espana,2025,https://www.procyclingstats.com/race/vuelta-a-...,NR,carlos-verona,",,0:00",1992-11-4,1.86,images/riders/my/dq/carlos-verona-2026.jpg,Carlos Verona,...,-,-,11,476.0,1521.0,21.0,6.0,1318.0,384.0,0.0
32728,vuelta-a-espana,2025,https://www.procyclingstats.com/race/vuelta-a-...,NR,orluis-aular,",,0:00",1996-11-5,1.76,images/riders/kb/dq/orluis-aular-2026.jpg,Orluis Aular,...,-,-,11,829.0,479.0,633.0,738.0,337.0,1322.0,0.0
32729,vuelta-a-espana,2025,https://www.procyclingstats.com/race/vuelta-a-...,NR,carlos-canal,",,0:00",2001-6-28,1.79,images/riders/kb/dq/carlos-canal-2026.png,Carlos Canal,...,-,-,11,587.0,415.0,115.0,158.0,214.0,640.0,0.0


In [32]:
# Speichern des aktuellen Datenbestands in Pickle Format .pckl

output_dir = '../../data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Den bereinigten DataFrame speichern
file_path = os.path.join(output_dir, '01_cleaned_master_data.pkl')
df_final.to_pickle(file_path)

print(f"Checkpoint gesetzt! Datei gespeichert unter: {file_path}")
print(f"Datensatz enthält jetzt {df_final.shape[1]} Spalten und {df_final.shape[0]} Zeilen.")

Checkpoint gesetzt! Datei gespeichert unter: ../../data/processed\01_cleaned_master_data.pkl
Datensatz enthält jetzt 33 Spalten und 32731 Zeilen.


## **Weiter mit Explorations Notebook** (neu)

Daten dann folgendermaßen laden:
```
import pandas as pd
df = pd.read_pickle('../../data/processed/01_cleaned_master_data.pkl')
```

### Offene Spalten zum extrahieren:
- points per season -> Mit Etappenjahr mappen
- Team History -> mit Etappenjahr mappen

### Weitere To Do's
- Einheiten rausnehmen aus Spalten (Dokumentieren!)


- Ausprägungen in den Spalten analysieren (Zahlen, Diagramme)
- NAs prüfen + Umgang
- Säubern der Spalten die wir nicht brauchen
- Doku schreiben Spaltennamen
- Wetter API 
    - Koordinaten (über Start und Zielort)
    - Best Practise Umsetzung Datengestalt/ -Umfang